# 🛡️ RAG 첫 실습: 보안 이벤트 → 문서 검색 → LLM 분석 → HTML 위협보고서

RAG가 무엇인지 직접 확인합니다. **짧은 보안 문서 5개**를 만들어 검색하고, 검색 결과를 **Qwen LLM에 전달**한 뒤, 마지막에 **완성된 HTML 위협보고서 파일**을 생성합니다.

**전체 흐름:** `보안 이벤트 1건 → TF-IDF 검색 → 관련 문서 1개 → Qwen LLM → HTML 보고서`  
⚠️ 이벤트와 문서는 **교육용 가상 자료**입니다. 실제 침해사고로 단정하지 않습니다.


## 1. 라이브러리 설치

- `scikit-learn`: 문서 검색을 위한 TF-IDF와 코사인 유사도
- `transformers`, `torch`: 무료 Qwen LLM 실행

In [26]:
!pip -q install -U scikit-learn transformers accelerate torch pandas

## 2. 예제 보안 이벤트 만들기 — RAG에게 물어볼 내용

이상탐지 모델을 학습하지 않고, 먼저 **DDoS로 의심되는 이벤트 1건**을 예제로 사용합니다. 다음 실습에서는 이 부분을 실제 이상탐지 결과로 교체합니다.

In [27]:
event = {
    "event_id": "DEMO-001",
    "timestamp": "2026-11-19 10:30:00",
    "src_ip": "198.51.100.130",   # 문서용 예시 IP
    "dst_ip": "10.10.5.10",
    "suspected_attack": "DDoS",  # 교육용 가상 탐지 결과
    "packet_count": 12000,
    "byte_count": 800000,
    "connection_duration": 30,
}

# 검색에 사용할 질문(문장)
query = (f"{event['suspected_attack']} 의심 이벤트. "
         f"짧은 시간 패킷 {event['packet_count']}개, "
         f"트래픽 {event['byte_count']}바이트, "
         "서비스 장애 및 대량 요청 점검 방법")
print("[검색 질문]", query)

[검색 질문] DDoS 의심 이벤트. 짧은 시간 패킷 12000개, 트래픽 800000바이트, 서비스 장애 및 대량 요청 점검 방법


## 3. RAG의 참고자료 만들기 — 작은 지식베이스 5개

`knowledge_base`는 AI가 **검색할 문서들**입니다. 지금은 직접 입력하지만 실제 업무에서는 매뉴얼·PDF·위협 인텔리전스·보고서 등으로 바꿀 수 있어요.

**핵심:** 지식베이스를 만든다고 LLM을 재학습시키는 건 아닙니다.

In [28]:
knowledge_base = [
    {"title": "Brute Force 대응", "attack": "Brute Force",
     "description": "반복적인 로그인 실패와 계정 비밀번호 무차별 대입을 점검한다.",
     "check": "동일 IP 로그인 실패 횟수, 계정 잠금, MFA 설정 확인",
     "response": "로그인 이력 조사, 계정 보호, 접근제어 강화"},
    {"title": "Port Scan 대응", "attack": "Port Scan",
     "description": "짧은 시간에 많은 목적지 포트로 연결을 시도하는 스캔 행위를 확인한다.",
     "check": "접근 포트 수, 방화벽/IDS 로그, 이후 취약점 접근 확인",
     "response": "불필요 포트 차단, 노출 서비스 및 후속 접속 점검"},
    {"title": "Data Exfiltration 대응", "attack": "Data Exfiltration",
     "description": "내부에서 외부로 대량 데이터를 전송하는 정보유출 징후를 확인한다.",
     "check": "송신 바이트 수, 목적지, DLP/프록시 기록 확인",
     "response": "관련 통신 및 파일 반출 경로 확인, 필요 시 전송 제한"},
    {"title": "C2 Communication 대응", "attack": "C2 Communication",
     "description": "감염 의심 단말과 외부 명령제어 서버의 주기적 통신을 확인한다.",
     "check": "DNS/HTTP 통신 주기, 목적지 도메인, 프로세스 확인",
     "response": "단말 점검, 필요 시 격리 및 의심 목적지 차단"},
    {"title": "DDoS 대응", "attack": "DDoS",
     "description": "짧은 시간에 대량 패킷 또는 요청이 서비스로 집중되어 가용성에 영향을 줄 수 있다.",
     "check": "패킷·요청 수 급증, 유입 출발지 분포, 서버 지연·오류율, 방화벽 로그 확인",
     "response": "트래픽 추이 분석, 레이트 리밋·방화벽 정책 검토, 서비스 상태 관찰"},
]

import pandas as pd
kb_df = pd.DataFrame(knowledge_base)
display(kb_df[["title", "attack", "description"]])

,title,attack,description
0,Brute Force 대응,Brute Force,반복적인 로그인 실패와 계정 비밀번호 무차별 대입을 점검한다.
1,Port Scan 대응,Port Scan,짧은 시간에 많은 목적지 포트로 연결을 시도하는 스캔 행위를 확인한다.
2,Data Exfiltration 대응,Data Exfiltration,내부에서 외부로 대량 데이터를 전송하는 정보유출 징후를 확인한다.
3,C2 Communication 대응,C2 Communication,감염 의심 단말과 외부 명령제어 서버의 주기적 통신을 확인한다.
4,DDoS 대응,DDoS,짧은 시간에 대량 패킷 또는 요청이 서비스로 집중되어 가용성에 영향을 줄 수 있다.


## 4. R — Retrieval: 관련 보안 문서 찾기

1. 질문과 보안 문서를 **TF-IDF 숫자 벡터**로 바꿉니다.
2. **코사인 유사도**를 계산합니다.
3. 가장 비슷한 문서 **1개**를 가져옵니다.

여기서는 *별도의 임베딩 API 없이* 검색하는 가장 단순한 RAG 형태예요.

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 공격 이름, 설명, 점검 항목을 한 문장으로 합치기
doc_texts = [
    f"{d['attack']} {d['title']} {d['description']} {d['check']} {d['response']}"
    for d in knowledge_base
]

# 한글에서도 비교적 쉽게 작동하도록 글자 단위 TF-IDF 사용
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
doc_vectors = vectorizer.fit_transform(doc_texts)
query_vector = vectorizer.transform([query])

scores = cosine_similarity(query_vector, doc_vectors)[0]
results = kb_df[["title", "attack"]].copy()
results["similarity"] = scores
results = results.sort_values("similarity", ascending=False)
display(results)

# 가장 비슷한 문서 1개 선택
best_idx = int(scores.argmax())
found_doc = knowledge_base[best_idx]
print("✅ 검색된 문서:", found_doc["title"])
print("✅ 문서 내용:", found_doc["description"])

,title,attack,similarity
4,DDoS 대응,DDoS,0.423065
1,Port Scan 대응,Port Scan,0.131674
3,C2 Communication 대응,C2 Communication,0.106867
2,Data Exfiltration 대응,Data Exfiltration,0.058311
0,Brute Force 대응,Brute Force,0.025809


✅ 검색된 문서: DDoS 대응
✅ 문서 내용: 짧은 시간에 대량 패킷 또는 요청이 서비스로 집중되어 가용성에 영향을 줄 수 있다.


## 5. A — Augmentation: 검색 문서를 LLM 질문에 추가

같은 모델에 **이벤트 정보만** 주는 경우와 **검색한 문서까지** 주는 경우를 비교해 보세요. 문서가 없는 답변은 필요한 정보를 스스로 추측할 수도 있습니다.

In [30]:
# 문서를 붙이지 않은 질문
prompt_without_rag = f"""다음 보안 이벤트를 3문장으로 설명해 주세요.
이벤트: {query}
확인되지 않은 침해 사실은 단정하지 마세요."""

# 검색된 문서를 포함한 질문 (RAG)
prompt_with_rag = f"""당신은 보안 관제 분석가입니다.
다음 [이벤트]와 [참고문서]에 근거해 한국어로 짧은 위협 분석을 작성하세요.
실제 공격 성공 여부는 확인되지 않았습니다. 근거 없는 추측은 하지 마세요.

[이벤트]
ID: {event['event_id']}
시각: {event['timestamp']}
출발지: {event['src_ip']}
목적지: {event['dst_ip']}
의심 공격: {event['suspected_attack']}
패킷 수: {event['packet_count']}
바이트 수: {event['byte_count']}
연결 시간(초): {event['connection_duration']}

[검색된 참고문서]
제목: {found_doc['title']}
설명: {found_doc['description']}
점검 항목: {found_doc['check']}
권고 대응: {found_doc['response']}

[작성 형식]
1. 이벤트 요약
2. 주요 의심 근거
3. 초기 대응 권고
4. 추가 확인 사항
각 항목을 1~2문장으로 작성하세요."""

print("RAG 프롬프트 미리보기:\n")
print(prompt_with_rag)

RAG 프롬프트 미리보기:

당신은 보안 관제 분석가입니다.
다음 [이벤트]와 [참고문서]에 근거해 한국어로 짧은 위협 분석을 작성하세요.
실제 공격 성공 여부는 확인되지 않았습니다. 근거 없는 추측은 하지 마세요.

[이벤트]
ID: DEMO-001
시각: 2026-11-19 10:30:00
출발지: 198.51.100.130
목적지: 10.10.5.10
의심 공격: DDoS
패킷 수: 12000
바이트 수: 800000
연결 시간(초): 30

[검색된 참고문서]
제목: DDoS 대응
설명: 짧은 시간에 대량 패킷 또는 요청이 서비스로 집중되어 가용성에 영향을 줄 수 있다.
점검 항목: 패킷·요청 수 급증, 유입 출발지 분포, 서버 지연·오류율, 방화벽 로그 확인
권고 대응: 트래픽 추이 분석, 레이트 리밋·방화벽 정책 검토, 서비스 상태 관찰

[작성 형식]
1. 이벤트 요약
2. 주요 의심 근거
3. 초기 대응 권고
4. 추가 확인 사항
각 항목을 1~2문장으로 작성하세요.


## 6. G — Generation: 무료 Qwen LLM 실행

이제 **실제로 LLM이 텍스트를 생성**합니다. 첫 실행 시 Hugging Face에서 모델이 다운로드되어 인터넷 연결이 필요해요. **API Key는 필요하지 않습니다.**

모델: `Qwen/Qwen2.5-0.5B-Instruct` (입문용으로 작은 모델이어서 답변 품질에 한계가 있을 수 있음)

In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPU에서는 float16, CPU에서는 float32
use_gpu = torch.cuda.is_available()
dtype = torch.float16 if use_gpu else torch.float32
llm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
llm = llm.to("cuda" if use_gpu else "cpu")
llm.eval()
print("✅ Qwen 준비 완료 / 실행 장치:", "GPU" if use_gpu else "CPU")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Qwen 준비 완료 / 실행 장치: CPU


In [ ]:
def ask_llm(prompt, max_new_tokens=300):

    # 1. 입력 문장 토큰화
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm.device)

    # 2. LLM 답변 생성
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # 3. 생성된 답변만 추출
    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    # 4. 토큰을 다시 텍스트로 변환
    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


print("[① RAG 없이 생성한 답변]")
answer_no_rag = ask_llm(prompt_without_rag)
print(answer_no_rag)

print("\n[② RAG 적용 후 생성한 답변]")
answer_rag = ask_llm(prompt_with_rag)
print(answer_rag)

[① RAG 없이 생성한 답변]


## 7. 최종 HTML 위협보고서 만들기

**보고서 한 장에 모두 담습니다.** 이벤트 원본, RAG 검색 결과, 위협 근거, 점검 및 대응 항목, **Qwen이 작성한 내용**, 비교용 RAG 미적용 답변까지 포함합니다.
[링크 텍스트](https://)

In [51]:

from IPython.display import HTML, display
from html import escape
from datetime import datetime
from pathlib import Path

# ✅ HTML에 들어갈 데이터 안전하게 처리
safe = lambda value: escape(str(value))

# ✅ 최종 HTML 위협보고서 생성
html_report = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>교육용 RAG 위협 분석 보고서</title>

<style>
* {{ box-sizing: border-box; }}

body {{
    margin: 0;
    font-family: 'Malgun Gothic', Arial, sans-serif;
    background: #f1f5f9;
    color: #17253a;
    line-height: 1.65;
}}

main {{
    max-width: 1000px;
    margin: 32px auto;
    padding: 0 18px 36px;
}}

header {{
    background: #142841;
    color: white;
    border-radius: 16px;
    padding: 34px 36px;
}}

header h1 {{
    margin: 0 0 5px;
    font-size: 26px;
}}

header p {{
    margin: 0;
    color: #bfd2e4;
}}

.section {{
    margin-top: 18px;
    background: white;
    padding: 25px 28px;
    border-radius: 14px;
    box-shadow: 0 2px 12px #182b430c;
}}

h2 {{
    margin: 0 0 15px;
    font-size: 18px;
    color: #163658;
}}

.grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(190px, 1fr));
    gap: 12px;
}}

.field {{
    background: #f6f9fd;
    border: 1px solid #e6edf5;
    border-radius: 9px;
    padding: 11px 14px;
}}

.label {{
    font-size: 12px;
    color: #5b6c81;
}}

.value {{
    font-weight: 700;
    overflow-wrap: anywhere;
}}

.tag {{
    display: inline-block;
    background: #e7effa;
    color: #15518e;
    padding: 4px 10px;
    border-radius: 99px;
    font-size: 12px;
    font-weight: bold;
}}

pre {{
    white-space: pre-wrap;
    overflow-wrap: anywhere;
    background: #f6f9fd;
    padding: 17px 19px;
    border-radius: 10px;
    font: 14px/1.75 'Malgun Gothic', Arial, sans-serif;
}}

.warning {{
    border-left: 4px solid #d99d34;
    padding: 12px 15px;
    background: #fff8ea;
    border-radius: 5px;
}}

.small {{
    font-size: 12px;
    color: #697b90;
}}

footer {{
    margin: 24px 0;
    text-align: center;
    color: #60758a;
    font-size: 12px;
}}

@media print {{
    body {{ background: white; }}
    main {{ margin: 0; max-width: none; }}
    .section {{
        box-shadow: none;
        border: 1px solid #e5e7eb;
        break-inside: avoid;
    }}
}}
</style>
</head>

<body>
<main>

<header>
<h1>🛡️ 위협 분석 보고서</h1>
<p>보안 이벤트 → RAG 근거 검색 → Qwen LLM 분석 | 교육용 예시</p>
<p>생성 시각: {safe(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))}</p>
</header>

<section class="section">
<h2>01. 이벤트 개요 <span class="tag">분석 필요 · 공격 미확정</span></h2>

<div class="grid">
<div class="field">
    <div class="label">이벤트 ID</div>
    <div class="value">{safe(event['event_id'])}</div>
</div>

<div class="field">
    <div class="label">발생 시각</div>
    <div class="value">{safe(event['timestamp'])}</div>
</div>

<div class="field">
    <div class="label">의심 공격</div>
    <div class="value">{safe(event['suspected_attack'])}</div>
</div>

<div class="field">
    <div class="label">출발지 → 목적지</div>
    <div class="value">{safe(event['src_ip'])} → {safe(event['dst_ip'])}</div>
</div>

<div class="field">
    <div class="label">패킷 수</div>
    <div class="value">{event['packet_count']:,}</div>
</div>

<div class="field">
    <div class="label">바이트 수</div>
    <div class="value">{event['byte_count']:,}</div>
</div>

<div class="field">
    <div class="label">연결 시간</div>
    <div class="value">{event['connection_duration']}초</div>
</div>
</div>
</section>

<section class="section">
<h2>02. RAG 검색 근거</h2>

<p>
<strong>검색된 문서:</strong>
{safe(found_doc['title'])}
<span class="tag">유사도 {scores[best_idx]:.3f}</span>
</p>

<p><strong>공격 설명:</strong> {safe(found_doc['description'])}</p>

<p class="small">
유사도는 TF-IDF 기반 텍스트 일치 정도이며, 공격 발생 확률이 아닙니다.
</p>
</section>

<section class="section">
<h2>03. 주요 확인사항</h2>
<p>{safe(found_doc['check'])}</p>
</section>

<section class="section">
<h2>04. 권고 초기 대응</h2>
<p>{safe(found_doc['response'])}</p>
</section>

<section class="section">
<h2>05. Qwen LLM 분석 (RAG 적용)</h2>

<pre>{safe(answer_rag)}</pre>

<p class="small">
생성형 모델의 출력에는 오류가 있을 수 있습니다.
위 근거와 원본 이벤트로 교차 검증하세요.
</p>
</section>

<section class="section">
<h2>06. 비교 학습: RAG 미적용 답변</h2>

<pre>{safe(answer_no_rag)}</pre>
</section>

</main>
</body>
</html>"""

# ✅ HTML 파일 저장
file_name = "RAG_위협분석_보고서.html"
Path(file_name).write_text(html_report, encoding="utf-8")

print("✅ HTML 보고서 생성:", file_name)

# ✅ Colab 화면에 보고서 출력
display(HTML(html_report))


✅ HTML 보고서 생성: RAG_위협분석_보고서.html


## 8. HTML 보고서 다운로드

Colab 왼쪽 파일 메뉴에서도 다운로드할 수 있고, 아래 코드를 실행하면 브라우저로 파일을 내려받을 수 있어요. 파일을 열면 **단독 HTML 보고서**로 표시됩니다.

In [ ]:
from IPython.display import HTML, display
from google.colab import files
from pathlib import Path
import re

# Qwen이 코드블록을 붙였을 경우 제거
html_report = re.sub(
    r"^```(?:html)?\s*|\s*```$",
    "",
    html_report.strip(),
    flags=re.IGNORECASE
)

# HTML 파일 저장
file_name = "RAG_위협보고서.html"
Path(file_name).write_text(html_report, encoding="utf-8")

# 화면 출력
display(HTML(html_report))

# 다운로드
files.download(file_name)

---
### 교육생에게 설명할 핵심 3문장

1. **R(검색)**: 관련 보안 문서를 TF-IDF와 코사인 유사도로 찾는다.
2. **A(정보 보강)**: 찾은 문서를 질문과 함께 LLM에 전달한다.
3. **G(생성)**: Qwen LLM이 검색된 근거를 참고해 분석 글을 만들고 HTML로 정리한다.

**다음 실습 연결:** 지금은 `event`를 직접 입력했지만, 다음 노트북에서는 `pd.read_csv()`로 NSL-KDD를 불러와 Isolation Forest가 뽑은 이벤트로 교체할 수 있습니다.